<h2>Exercise 09: Plotly</h2>

In [ ]:
import pandas as pd
import sqlite3
import plotly.graph_objects as go
import numpy as np

### Preprocessing

In [2]:
conn = sqlite3.connect("../data/checking-logs.sqlite")

In [13]:
df  = pd.io.sql.read_sql(
    """
    SELECT
        *
    FROM checker
    WHERE 
        labname LIKE "project1" 
        AND uid LIKE "user_%"
        AND status = "ready"
    """,
    conn,
    parse_dates=["timestamp"]
)

df

,index,status,success,timestamp,numTrials,labname,uid
0,7,ready,0,2020-04-17 05:19:02.744528,1,project1,user_4
1,9,ready,1,2020-04-17 05:22:45.549397,2,project1,user_4
2,11,ready,1,2020-04-17 05:34:24.422370,3,project1,user_4
3,13,ready,0,2020-04-17 05:43:27.773992,4,project1,user_4
4,15,ready,1,2020-04-17 05:46:32.275104,5,project1,user_4
...,...,...,...,...,...,...,...
946,3186,ready,1,2020-05-15 10:22:39.698523,26,project1,user_19
947,3187,ready,0,2020-05-15 10:22:46.248162,27,project1,user_19
948,3189,ready,1,2020-05-15 10:23:18.043212,28,project1,user_19
949,3191,ready,1,2020-05-15 10:38:14.430013,27,project1,user_28


In [14]:
data_matrix = pd.crosstab(df['timestamp'].dt.date, df['uid']).cumsum()

all_dates = data_matrix.index
all_users = data_matrix.columns

In [16]:
conn.close()

### Result

In [15]:
frames = []
for date in all_dates:
    frame_data = []
    for user in all_users:
        # Срез данных до текущей даты
        sub_data = data_matrix.loc[:date, user]
        
        frame_data.append(go.Scatter(
            x=sub_data.index,    # ИСПОЛЬЗУЕМ ДАТЫ
            y=sub_data.values,
            mode='lines+markers', # Линии + Точки
            name=user,
            line=dict(width=2),
            marker=dict(size=6)
        ))
    frames.append(go.Frame(data=frame_data, name=str(date)))

# 3. НАЧАЛЬНЫЕ ДАННЫЕ (Первая дата)
initial_data = []
first_date = all_dates[0]
for user in all_users:
    sub_data = data_matrix.loc[:first_date, user]
    initial_data.append(go.Scatter(
        x=sub_data.index,
        y=sub_data.values,
        mode='lines+markers',
        name=user,
        line=dict(width=2),
        marker=dict(size=6)
    ))

# 4. НАСТРОЙКИ (LAYOUT)
layout = go.Layout(
    title='Dynamic of commits per user in project1',

    xaxis=dict(
        title='timestamp',
        range=[all_dates[0], all_dates[-1]], # Фиксируем диапазон дат
        showgrid=True,
        gridcolor='white'
    ),
    yaxis=dict(
        title='numTrials',
        range=[0, data_matrix.max().max() * 1.05], # Фиксируем высоту
        showgrid=True, 
        gridcolor='white'
    ),
    plot_bgcolor='rgb(230, 230, 230)', 
    paper_bgcolor='white',
    
    # Кнопка Play
    updatemenus=[
        dict(
            type="buttons",
            buttons=[
                dict(label="Play",
                     method="animate",
                     args=[None, dict(frame=dict(duration=200, redraw=True), fromcurrent=True)]), # duration - скорость
                dict(label="Pause",
                     method="animate",
                     args=[[None], dict(frame=dict(duration=0, redraw=False), mode="immediate", transition=dict(duration=0))])
            ],
            direction="left",
            pad={"r": 10, "t": 87},
            showactive=False,
            x=0.1,
            y=0,
            xanchor="right",
            yanchor="top"
        )
    ]
)

fig = go.Figure(data=initial_data, layout=layout, frames=frames)
fig.show()